# 2d — Koszul bracket Leibniz: $[\omega, f\eta]_{T^*M} = f\,[\omega, \eta]_{T^*M} + (\pi^\sharp(\omega)f)\,\eta$

**Problem (d).** $T^*M$ üzerinde Koszul bracket için sağ-Leibniz kuralı:

$$
[\omega, f\eta]_{T^*M} = f\,[\omega, \eta]_{T^*M} + \bigl(\pi^\sharp(\omega)\,f\bigr)\,\eta,
$$

Koszul bracket'in tanımı:

$$
[\alpha, \beta]_K \;=\; \mathcal{L}_{\pi^\sharp\alpha}\beta \;-\; \mathcal{L}_{\pi^\sharp\beta}\alpha \;-\; d\langle \pi^\sharp\alpha, \beta \rangle.
$$

$\pi^\sharp$ bir tensör (C^∞-lineer) ve $\pi$ anti-simetrik olduğu için $\langle\pi^\sharp\alpha,\beta\rangle + \langle\pi^\sharp\beta,\alpha\rangle = 0$. Bu iki olgu birlikte sağ-Leibniz'i kapatır.

## Strateji — niye 2a/b/c'den niteliksel olarak farklı

2a/b/c **operatör-seviyesi** kimliklerdi (Cartan magic, $d^2=0$, pairing). 2d ise **$C^\infty$-modül cebri** istiyor: $f\eta$ ürününün üç ayrı geometrik operatör altında nasıl davrandığı:

| Terim | Açılan kimlik | Engine'de var mı? |
|---|---|---|
| $\mathcal{L}_{X_\omega}(f\eta)$ | $X_\omega(f)\,\eta + f\,\mathcal{L}_{X_\omega}(\eta)$ | **Otomatik** — Cartan magic + product-rule |
| $\mathcal{L}_{\pi^\sharp(f\eta)}\omega$ | $f\,\mathcal{L}_{X_\eta}(\omega) + df\wedge\iota_{X_\eta}(\omega)$ | **Yok** — $\pi^\sharp$ $C^\infty$-lineer + $\mathcal{L}_{fX}$ açılımı |
| $d\langle X_\omega, f\eta\rangle$ | $df\cdot\langle X_\omega,\eta\rangle + f\cdot d\langle X_\omega,\eta\rangle$ | **Otomatik** — $d$ Leibniz + pairing C^∞-linearity (axiom) |
| $df$ ile $df\cdot\langle X_\omega,\eta\rangle$ kapanışı | $\iota_{X_\eta}(\omega) = -\langle X_\omega,\eta\rangle$ ($\pi$ anti-sym) | **Yok** — Musical compatibility rank-2 (Faz 12 #8) |

İki gerçek geometrik girdi: **(i)** $\pi^\sharp$ $C^\infty$-lineer + $\mathcal{L}_{fX}$ kuralı, **(ii)** $\pi$ anti-simetrik. İkisini inline aksiyom olarak koyuyoruz; gerisini engine kapatıyor.

**Mode disiplini.** Cartan-mode $\mathcal{L}_X$ engine tarafından her gördüğünde $d\circ\iota_X + \iota_X\circ d$'ye açılır. Bu davranışı $X_\omega$ için **istiyoruz** (LHS'taki $\mathcal{L}_{X_\omega}(f\eta)$ açılsın), ama $X_\eta$ için **istemiyoruz** ($\mathcal{L}_{X_\eta}(\omega)$ opak kalmalı, çünkü $\eta$ ve $\omega$ generic 1-formlar — Cartan açılımı $d(\omega)$, $d(\eta)$ üretir, bunlar daha fazla indirgenemez ve zincir tıkanır). Çözüm: $X_\omega$ Cartan, $X_\eta$ flow.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Kurulum

- $f$ — fonksiyon (0-form).
- $\omega, \eta$ — generic 1-formlar.
- $X_\omega, X_\eta$ — sırasıyla $\pi^\sharp(\omega)$, $\pi^\sharp(\eta)$. Sharp'ın **bir tensör (derivation değil)** olması, sistem-seviyesinde derecelendirmeyi açıkça takip etmemizi gerektirir; bu yüzden $X_\omega$ ve $X_\eta$'yi **isimli `Derivation`** olarak kuruyoruz, sharp'ın kendisini $T^*M\to TM$ izomorfizmi olarak tutmuyoruz (şu anki framework'te `Sharp` Derivation atomu, ama 1-form üzerine uygulandığında çıktı derecesi tutarsız oluyor — bkz. Faz 12 #6).
- $\mathcal{L}_{X_\omega}$ **cartan-mode**, $\mathcal{L}_{X_\eta}$ **flow-mode**.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative, LieDerivative
from jacopy.calculus.pairing import pairing, Pairing
from jacopy.core.expr import Expr, Integer, Neg, Product, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
omega = Symbol("ω")
eta = Symbol("η")
reg.declare(f, Graded(degree=0))
reg.declare(omega, Graded(degree=1))
reg.declare(eta, Graded(degree=1))

# π^♯(ω) ve π^♯(η) — birer vektör alanı (degree-0 derivation).
X_om = Derivation("X_ω", degree=0)
X_et = Derivation("X_η", degree=0)

L_Xom = lie_derivative(X_om, definition="cartan")
L_Xet = lie_derivative(X_et, definition="flow")

# Koszul bracket'i isimli Symbol olarak kuruyoruz; tanımı aşağıda axiom.
koszul_om_feta = Symbol("[ω,fη]_K")
koszul_om_eta  = Symbol("[ω,η]_K")
reg.declare(koszul_om_feta, Graded(degree=1))
reg.declare(koszul_om_eta,  Graded(degree=1))

print("f, ω, η        :", f, omega, eta)
print("X_ω (cartan)   :", X_om, " L =", L_Xom)
print("X_η (flow)     :", X_et, " L =", L_Xet)
print("[ω,fη]_K       :", koszul_om_feta)
print("[ω,η]_K        :", koszul_om_eta)

f, ω, η        : f ω η
X_ω (cartan)   : X_ω  L = L_X_ω
X_η (flow)     : X_η  L = L_X_η
[ω,fη]_K       : [ω,fη]_K
[ω,η]_K        : [ω,η]_K


## 2. Aksiyomlar (4 tane)

**A-K1 — `[ω, fη]_K` defining (with sharp $C^\infty$-linearity folded).** Saf Koszul tanımı $\mathcal{L}_{X_\omega}(f\eta) - \mathcal{L}_{\pi^\sharp(f\eta)}\omega - d\langle X_\omega, f\eta\rangle$ olurdu; $\pi^\sharp(f\eta) = f\cdot X_\eta$ ($C^\infty$-lineerlik) ve $\mathcal{L}_{fX}\omega = f\,\mathcal{L}_X\omega + df\wedge\iota_X\omega$ olduğu için, ortadaki terimi önceden bu açılmış formuyla yazıyoruz:

$$
[\omega, f\eta]_K \;\to\; \mathcal{L}_{X_\omega}(f\eta) - f\,\mathcal{L}_{X_\eta}(\omega) - df\wedge\iota_{X_\eta}(\omega) - d\langle X_\omega, f\eta\rangle.
$$

Bu yaklaşım, sistem $\pi^\sharp$'i $C^\infty$-lineer olarak tanımadığı için pratik gereklilik. **Faz 12** (#6 multilinear evaluation, #8 musical compatibility bilinear) landing ettiğinde bu fold edilmiş hâl iki ayrı satıra (sharp linearity + $L_{fX}$) bölünebilir.

**A-K2 — `[ω, η]_K` defining.** Standart Koszul:

$$
[\omega, \eta]_K \;\to\; \mathcal{L}_{X_\omega}(\eta) - \mathcal{L}_{X_\eta}(\omega) - d\langle X_\omega, \eta\rangle.
$$

**A-pairing — pairing $C^\infty$-lineer.** $\langle X_\omega, f\eta\rangle = f\,\langle X_\omega, \eta\rangle$.

**A-π-antisym — $\pi$ anti-simetri.** $\iota_{X_\eta}(\omega) = -\langle X_\omega, \eta\rangle$. Çünkü $\iota_{\pi^\sharp\eta}\omega = \omega(\pi^\sharp\eta) = \pi(\eta,\omega) = -\pi(\omega,\eta) = -\langle\pi^\sharp\omega, \eta\rangle = -\langle X_\omega,\eta\rangle$.

In [3]:
class KoszulDef_om_feta(Definition):
    """A-K1: [ω, fη]_K defining (sharp C∞-linearity + L_{fX} folded)."""
    name = "[ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩"

    def matches(self, expr):
        return expr == koszul_om_feta

    def rewrite(self, expr):
        return Sum(
            Act(L_Xom, Product(f, eta)),
            Neg(Product(f, Act(L_Xet, omega))),
            Neg(Product(Act(d, f), Act(interior(X_et), omega))),
            Neg(Act(d, pairing(X_om, Product(f, eta)))),
        )


class KoszulDef_om_eta(Definition):
    """A-K2: [ω, η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩."""
    name = "[ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩"

    def matches(self, expr):
        return expr == koszul_om_eta

    def rewrite(self, expr):
        return Sum(
            Act(L_Xom, eta),
            Neg(Act(L_Xet, omega)),
            Neg(Act(d, pairing(X_om, eta))),
        )


class PairingXomFeta(Definition):
    """A-pairing: ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩."""
    name = "⟨X_ω, fη⟩ = f·⟨X_ω, η⟩"

    def matches(self, expr):
        return (
            isinstance(expr, Pairing)
            and expr.alpha == X_om
            and expr.X == Product(f, eta)
        )

    def rewrite(self, expr):
        return Product(f, pairing(X_om, eta))


class PiAntiSymm(Definition):
    """A-π-antisym: ι_{X_η}(ω) = −⟨X_ω, η⟩."""
    name = "ι_{X_η}(ω) = −⟨X_ω, η⟩"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, InteriorProduct)
            and expr.op.vector_field == X_et
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Neg(pairing(X_om, eta))


for axiom in (KoszulDef_om_feta, KoszulDef_om_eta, PairingXomFeta, PiAntiSymm):
    print("-", axiom.name)

- [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩
- [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩
- ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩
- ι_{X_η}(ω) = −⟨X_ω, η⟩


## 3. Engine

`default_engine` Cartan magic'i, $d^2=0$'ı, pairing'i ($\iota_X(df) = X(f)$), $\iota_X(f)=0$'ı zaten taşır; ayrıca product-rule ($d$ ve $\iota$ Leibniz'i) canonical-form pipeline'ında otomatik. Dört problem-aksiyomunu üzerine ekliyoruz.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(KoszulDef_om_feta())
engine.register(KoszulDef_om_eta())
engine.register(PairingXomFeta())
engine.register(PiAntiSymm())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩
 - [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩
 - ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩
 - ι_{X_η}(ω) = −⟨X_ω, η⟩


## 4. Hedef ve ispat

$$
[\omega, f\eta]_K \;=\; f\,[\omega, \eta]_K + (X_\omega(f))\,\eta.
$$

RHS'taki $X_\omega(f) = (\pi^\sharp\omega)(f)$ — Hamiltonian-türü skalar fonksiyon (vektör alanının fonksiyona uygulanması).

In [5]:
lhs = koszul_om_feta
rhs = Sum(
    Product(f, koszul_om_eta),
    Product(Act(X_om, f), eta),
)

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nKAPANDI — {len(chain)} adım.")

LHS: [ω,fη]_K
RHS: ((f * [ω,η]_K) + (X_ω(f) * η))

KAPANDI — 13 adım.


## 5. İspat zinciri — LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

\begin{align*}
[\omega,f\eta]_K &\to L_{X_}\omega\!\left(f \, \eta\right) - f \, L_{X_}\eta\!\left(\omega\right) - d\!\left(f\right) \, \iota_{X_}\eta\!\left(\omega\right) - d\!\left(\langle X_\omega,\, f \, \eta \rangle\right) && \text{[[\ensuremath{\omega},f\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(f\ensuremath{\eta}) − f\ensuremath{\cdot}L\_X\ensuremath{\eta}(\ensuremath{\omega}) − df\ensuremath{\cdot}\ensuremath{\iota}\_X\ensuremath{\eta}(\ensuremath{\omega}) − d\ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\omega},f\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(f\ensuremath{\eta}) − f\ensuremath{\cdot}L\_X\ensuremath{\eta}(\ensuremath{\omega}) − df\ensuremath{\cdot}\ensuremath{\iota}\_X\ensuremath{\eta}(\ensuremath{\omega}) − d\ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle}} \\
L_{X_}\omega\!\left(f \, \eta\right) &\to \left(d \, \iota_{X_}\omega\right)\!\left(f \, \eta\right) + \left(\iota_{X_}\omega \, d\right)\!\left(f \, \eta\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\iota_{X_}\eta\!\left(\omega\right) &\to -\langle X_\omega,\, \eta \rangle && \text{[\ensuremath{\iota}\_{X\_\ensuremath{\eta}}(\ensuremath{\omega}) = −\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{X\_\ensuremath{\eta}}(\ensuremath{\omega}) = −\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}} \\
\langle X_\omega,\, f \, \eta \rangle &\to f \, \langle X_\omega,\, \eta \rangle && \text{[\ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}} \\
[\omega,\eta]_K &\to L_{X_}\omega\!\left(\eta\right) - L_{X_}\eta\!\left(\omega\right) - d\!\left(\langle X_\omega,\, \eta \rangle\right) && \text{[[\ensuremath{\omega},\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(\ensuremath{\eta}) − L\_X\ensuremath{\eta}(\ensuremath{\omega}) − d\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\omega},\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(\ensuremath{\eta}) − L\_X\ensuremath{\eta}(\ensuremath{\omega}) − d\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}} \\
L_{X_}\omega\!\left(\eta\right) &\to \left(d \, \iota_{X_}\omega\right)\!\left(\eta\right) + \left(\iota_{X_}\omega \, d\right)\!\left(\eta\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(\left(\left(d \, \iota_{X_}\omega\right)\!\left(f \, \eta\right) + \left(\iota_{X_}\omega \, d\right)\!\left(f \, \eta\right)\right) - f \, L_{X_}\eta\!\left(\omega\right) - d\!\left(f\right) \, \left(-\langle X_\omega,\, \eta \rangle\right) - d\!\left(f \, \langle X_\omega,\, \eta \rangle\right)\right) - \left(f \, \left(\left(\left(d \, \iota_{X_}\omega\right)\!\left(\eta\right) + \left(\iota_{X_}\omega \, d\right)\!\left(\eta\right)\right) - L_{X_}\eta\!\left(\omega\right) - d\!\left(\langle X_\omega,\, \eta \rangle\right)\right) + X_\omega\!\left(f\right) \, \eta\right) &\to \left(\left(\left(d\!\left(\iota_{X_}\omega\!\left(f\right)\right) \, \eta - \iota_{X

## 6. Adım adım

Zincirin akışı:

1. **A-K1** — LHS açılımı: $[\omega,f\eta]_K \to L_{X_\omega}(f\eta) - f\,L_{X_\eta}(\omega) - df\,\iota_{X_\eta}(\omega) - d\langle X_\omega, f\eta\rangle$.
2. **Cartan magic** — $L_{X_\omega}(f\eta) \to (d\iota_{X_\omega} + \iota_{X_\omega} d)(f\eta)$.
3. **A-π-antisym** — $\iota_{X_\eta}(\omega) \to -\langle X_\omega,\eta\rangle$.
4. **A-pairing** — $\langle X_\omega, f\eta\rangle \to f\cdot\langle X_\omega,\eta\rangle$.
5. **A-K2** — RHS'taki $[\omega,\eta]_K$ açılımı.
6. **Cartan magic** — $L_{X_\omega}(\eta)$ açılımı.
7. **product-rule** — büyük genişleme: $d$ ve $\iota$ Leibniz'i tüm $f\cdot(\ldots)$ ürünlerinde uygulanır; canonical form çıkar.
8-11. **$\iota_X(f)=0$ + pairing** — $\iota_{X_\omega}(f)$ sıfır, $\iota_{X_\omega}(df) = X_\omega(f)$.
12. **product-rule** — sıfırların temizlenmesi.
13. **simplify** — kanonik form kapanışı: tüm terimler ya birbirini götürür ya $0$'a düşer.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" ↦  {step.after}")
    print()

[1] [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩ [axiom]
    [ω,fη]_K
 ↦  (L_X_ω((f * η)) + (-(f * L_X_η(ω))) + (-(d(f) * ι_X_η(ω))) + (-d(⟨X_ω, (f * η)⟩)))

[2] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_ω((f * η))
 ↦  ((d * ι_X_ω)((f * η)) + (ι_X_ω * d)((f * η)))

[3] ι_{X_η}(ω) = −⟨X_ω, η⟩ [axiom]
    ι_X_η(ω)
 ↦  (-⟨X_ω, η⟩)

[4] ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩ [axiom]
    ⟨X_ω, (f * η)⟩
 ↦  (f * ⟨X_ω, η⟩)

[5] [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩ [axiom]
    [ω,η]_K
 ↦  (L_X_ω(η) + (-L_X_η(ω)) + (-d(⟨X_ω, η⟩)))

[6] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_ω(η)
 ↦  ((d * ι_X_ω)(η) + (ι_X_ω * d)(η))

[7] product-rule 
    ((((d * ι_X_ω)((f * η)) + (ι_X_ω * d)((f * η))) + (-(f * L_X_η(ω))) + (-(d(f) * (-⟨X_ω, η⟩))) + (-d((f * ⟨X_ω, η⟩)))) + (-((f * (((d * ι_X_ω)(η) + (ι_X_ω * d)(η)) + (-L_X_η(ω)) + (-d(⟨X_ω, η⟩)))) + (X_ω(f) * η))))
 ↦  (((((d(ι_X_ω(f)) * η) + (-(ι_X_ω(f) * d(η))) + (d(f) * ι_X_ω(η)) + (f * d(ι_X_ω(η)))) + ((ι_X_ω(d(f)) * η) + (-

## Sonuç

$$
\boxed{\;[\omega, f\eta]_K = f\,[\omega, \eta]_K + (\pi^\sharp(\omega)f)\,\eta.\;}
$$

**13-adımlı zincir, 4 inline aksiyom.** 2a/b/c'ye kıyasla nitel fark: 2d $C^\infty$-modül cebri istiyor — sharp tensoriyalliği, $L_{fX}$ açılımı, $\pi$ anti-simetri. Bu üç olgu **gerçek geometrik girdiler**, framework'ün bilemeyeceği şeyler.

**Bu pass'ten çıkan altyapı düzeltmesi:** `jacopy/algorithms/sort_product.py:_degree_of`, `Act(d, f)` gibi compound factor'ların derecesini çıkaramıyordu (sadece Scalar/Derivation/registry-Graded). Bu pass'te eklenen düzeltme `algebra.derivation.degree_of`'a fallback yapıyor; canonical-form pipeline artık bileşik factor'leri sorunsuz sıralıyor. Bu olmadan 2d notebook'u monkey-patch'siz çalışmıyordu.

**Faz 12 etiketleri.** İdealde aşağıdaki üç parça bu notebook'u daha temizleyecek:

- **#6 Multilinear evaluation + sharp linearity** — $\pi^\sharp(f\eta) = f\cdot\pi^\sharp(\eta)$ aksiyomu yerleşik olur, A-K1'in fold edilmiş hâli iki ayrı satıra bölünür.
- **#8 Musical compatibility bilinear** — A-π-antisym'i theorem'e indirir.
- **12.C(c) `register_hamiltonian_defining_relation`** + Sharp/Pairing $C^\infty$-linearity wrappers — A-K1, A-K2, A-pairing'i tek-satır API'a indirir.

Bu üç landing edene kadar 4-aksiyom paterni Koszul-ailesi notebook'larında standart kalır.